# 🎯 Day 02a: Logistic Regression — Binary Classification From Scratch

---

## 🎯 What You'll Master Today
- Sigmoid function and why it works
- Cross-entropy (log) loss derivation
- Gradient computation and update rule
- Logistic regression from scratch + sklearn
- Decision boundary plotting

**Goal:** Implement a binary classifier from scratch in an interview.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  LOGISTIC REGRESSION — THE MATH                                  ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Sigmoid:   σ(z) = 1 / (1 + e⁻ᶻ)     range: (0, 1)           ║
║                                                                   ║
║  Model:     P(y=1|x) = σ(Xw + b)                               ║
║                                                                   ║
║  Loss:      J = -(1/n) Σ [ylog(ŷ) + (1-y)log(1-ŷ)]          ║
║             (Binary Cross-Entropy)                                ║
║                                                                   ║
║  Gradient:  ∂J/∂w = (1/n) Xᵀ(σ(Xw) - y)                      ║
║             Same form as linear regression!                      ║
║                                                                   ║
║  Predict:   ŷ = 1 if σ(Xw + b) ≥ 0.5 else 0                  ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import numpy as np

# ============================================================
# Sigmoid Function
# ============================================================

def sigmoid(z):
    """σ(z) = 1 / (1 + e^(-z))"""
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))  # clip for numerical stability

# Key properties
z = np.array([-10, -2, 0, 2, 10])
print(f"z:       {z}")
print(f"σ(z):    {sigmoid(z).round(4)}")
print(f"\nσ(0) = 0.5 (decision boundary)")
print(f"σ(-∞) → 0, σ(+∞) → 1")
print(f"σ'(z) = σ(z)(1 - σ(z)) — derivative")

In [ ]:
# ============================================================
# Generate Binary Classification Data
# ============================================================

from sklearn.datasets import make_classification

np.random.seed(42)
X, y = make_classification(
    n_samples=300, n_features=2, n_redundant=0,
    n_informative=2, n_clusters_per_class=1, random_state=42
)

print(f"X shape: {X.shape}")
print(f"y distribution: 0→{(y==0).sum()}, 1→{(y==1).sum()}")

In [ ]:
# ============================================================
# ⭐ Logistic Regression — From Scratch
# ============================================================

class LogisticRegressionScratch:
    def __init__(self, lr=0.01, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.losses = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for i in range(self.n_iters):
            # Forward pass
            z = X @ self.weights + self.bias
            y_pred = sigmoid(z)
            
            # Binary cross-entropy loss
            eps = 1e-15  # avoid log(0)
            loss = -np.mean(y * np.log(y_pred + eps) + (1 - y) * np.log(1 - y_pred + eps))
            self.losses.append(loss)
            
            # Gradients (same form as linear regression!)
            dw = (1 / n_samples) * (X.T @ (y_pred - y))
            db = (1 / n_samples) * np.sum(y_pred - y)
            
            # Update
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
            if i % 200 == 0:
                acc = np.mean((y_pred >= 0.5) == y)
                print(f"Iter {i:4d}: loss={loss:.4f}, acc={acc:.3f}")
        
        return self
    
    def predict_proba(self, X):
        return sigmoid(X @ self.weights + self.bias)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

# Train
model = LogisticRegressionScratch(lr=0.1, n_iters=1000)
model.fit(X, y)

# Final accuracy
y_pred = model.predict(X)
accuracy = np.mean(y_pred == y)
print(f"\nFinal accuracy: {accuracy:.3f}")
print(f"Weights: {model.weights.round(3)}, Bias: {model.bias:.3f}")

In [ ]:
# ============================================================
# sklearn Comparison
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

sk_model = LogisticRegression(max_iter=1000)
sk_model.fit(X, y)

y_pred_sk = sk_model.predict(X)

print(f"sklearn acc:  {accuracy_score(y, y_pred_sk):.3f}")
print(f"Scratch acc:  {accuracy:.3f}")
print(f"\nsklearn w: {sk_model.coef_[0].round(3)}, b: {sk_model.intercept_[0]:.3f}")
print(f"Scratch w: {model.weights.round(3)}, b: {model.bias:.3f}")

print(f"\nClassification Report:\n{classification_report(y, y_pred_sk)}")

In [ ]:
# ============================================================
# Decision Boundary — How to Plot
# ============================================================

# The decision boundary: w₁x₁ + w₂x₂ + b = 0
# Solving for x₂: x₂ = -(w₁x₁ + b) / w₂

w1, w2 = model.weights
b = model.bias

x1_range = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 100)
x2_boundary = -(w1 * x1_range + b) / w2

print(f"Decision boundary equation: x₂ = -({w1:.3f}·x₁ + {b:.3f}) / {w2:.3f}")
print(f"\nAt x₁=0: x₂ = {(-b/w2):.3f}")
print(f"Slope: {(-w1/w2):.3f}")

# In a real interview, you might be asked to sketch this
# Key insight: the boundary is LINEAR (that's why it's 'logistic LINEAR regression')

---

## 🗺️ Logistic Regression Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  LOGISTIC REGRESSION — From Memory                               ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. z = X @ w + b                                                ║
║  2. ŷ = sigmoid(z) = 1 / (1 + e⁻ᶻ)                            ║
║  3. Loss = -mean(y·log(ŷ) + (1-y)·log(1-ŷ))                  ║
║  4. dw = (1/n) * Xᵀ(ŷ - y)      ← same as linear!            ║
║  5. w -= lr * dw, b -= lr * db                                  ║
║                                                                   ║
║  vs LINEAR REGRESSION:                                           ║
║    Same gradient form! Only difference:                          ║
║    - ŷ goes through sigmoid (not raw)                           ║
║    - Loss is BCE (not MSE)                                       ║
║    - Output is probability, threshold at 0.5                    ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Why sigmoid and not linear? | Squashes to (0,1) for probabilities. Linear gives unbounded values |
| 2 | Why cross-entropy not MSE? | MSE with sigmoid creates non-convex loss. BCE is convex → guaranteed convergence |
| 3 | What does the decision boundary look like? | Linear: w₁x₁ + w₂x₂ + b = 0. It's a line (2D), plane (3D), hyperplane (nD) |
| 4 | How to handle multi-class? | One-vs-Rest (OvR) or softmax regression (multinomial) |
| 5 | Logistic vs linear regression gradient? | Same form: (1/n)Xᵀ(ŷ-y). The ŷ computation differs (sigmoid vs raw) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • σ(z) = 1/(1+e⁻ᶻ), σ(0) = 0.5                       │
## │  • BCE loss, not MSE                                     │
## │  • Gradient = (1/n)Xᵀ(ŷ-y) — same as linear!           │
## │  • Decision boundary is linear                          │
## │  • eps=1e-15 to avoid log(0)                            │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **SVM** — max margin, support vectors, kernel trick